# U05_C2 — Local Alignment and Scoring
**Course:** Concepts of Bioinformatics (CBBI-501) · **Unit 5:** Sequence Alignment and Database Searching
**Type:** Course notebook · **Time:** 40 min · **Difficulty:** ★★☆
**Goes with:** Lectures L12–L13, Practical S6 (BLAST)
**Before this:** U05_C1

## 1. What you will learn
- **Implement** Smith–Waterman local alignment and understand why the zero-floor condition is biologically essential [LO3].
- **Choose** appropriate substitution matrices (BLOSUM62 vs PAM250) based on evolutionary divergence [LO3].
- **Apply** affine gap penalties (gap opening vs gap extension) to model biological indel events [LO3].
- **Compare** global vs local alignment scores when aligning full-length globins against short functional motifs [LO3].


## 2. Biological question
Proteins frequently share isolated conserved domains (e.g. haem-binding pocket, kinase domain, zinc finger) surrounded by divergent, unalignable flanking sequences. If you force an alignment across the entire sequence with global alignment, the score is penalized heavily by non-homologous regions. How does local alignment find the highest-scoring matching sub-region?


In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

NOTEBOOK_ID = "U05_C2"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / NOTEBOOK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

import Bio
print("Python", sys.version.split()[0], "| Biopython", Bio.__version__)

## 3. Concept 1: Smith–Waterman Local Alignment
### Theory
Smith and Waterman (1981) modified Needleman–Wunsch with two crucial differences:
1. **Zero floor**: If all paths leading to a cell yield negative scores, the cell resets to zero:
   $$H(i, j) = \max \begin{cases}
   0 & \text{(Reset / Start new local alignment)} \\
   H(i-1, j-1) + s(x_i, y_j) \\
   H(i-1, j) + d \\
   H(i, j-1) + d
   \end{cases}$$
2. **Traceback**: Starts at the **maximum score cell** anywhere in the matrix (not bottom-right) and terminates as soon as a cell with score `0` is reached.


In [ ]:
# Worked Example: Smith-Waterman Local Alignment implementation
import numpy as np

def smith_waterman(seq_x, seq_y, match_score=3, mismatch_score=-2, gap_penalty=-2):
    m, n = len(seq_x), len(seq_y)
    H = np.zeros((m + 1, n + 1), dtype=int)
    max_score = 0
    max_pos = (0, 0)
    
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            s = match_score if seq_x[i-1] == seq_y[j-1] else mismatch_score
            score = max(0, H[i-1, j-1] + s, H[i-1, j] + gap_penalty, H[i, j-1] + gap_penalty)
            H[i, j] = score
            if score > max_score:
                max_score = score
                max_pos = (i, j)
                
    return H, max_score, max_pos

seq_a = "TGTTACGG"
seq_b = "GGTTGACTA"
H_mat, best_score, best_coord = smith_waterman(seq_a, seq_b)

print("Best Local Alignment Score:", best_score)
print("Best Score Located at Matrix Cell:", best_coord)


## 4. Concept 2: Substitution Matrices (BLOSUM and PAM)
### Theory
Amino acids have distinct physicochemical properties; substituting a Leucine with Isoleucine is far more conservative than substituting with a Proline.
- **PAM (Point Accepted Mutation)**: Margaret Dayhoff (1978). Derived from global alignments of closely related proteins ($>85\%$ identity). **Higher PAM number = greater evolutionary divergence** (e.g. PAM250 for distant relatives).
- **BLOSUM (Blocks Substitution Matrix)**: Henikoff & Henikoff (1992). Derived from local ungapped blocks of conserved protein families. **Higher BLOSUM number = closer evolutionary relatedness** (e.g. BLOSUM80 for close, BLOSUM62 standard default, BLOSUM45 for distant).


In [ ]:
# Worked Example: Inspecting BLOSUM62 log-odds scores in Biopython
from Bio.Align import substitution_matrices

blosum62 = substitution_matrices.load("BLOSUM62")

# Check identical match scores
print("Trp self-match (W-W):", blosum62["W", "W"])  # Rare, structurally rigid -> high score (+11)
print("Ala self-match (A-A):", blosum62["A", "A"])  # Common, simple (+4)

# Check conservative vs radical substitutions
print("Glu -> Asp (E-D, both negative acidic):", blosum62["E", "D"])  # Conservative (+2)
print("Glu -> Val (E-V, sickle-cell mutation): ", blosum62["E", "V"])  # Non-conservative (-2)
print("Trp -> Cys (W-C, radical mismatch):    ", blosum62["W", "C"])  # Disruptive (-2)


## 5. Concept 3: Affine Gap Penalties
### Theory
Biologically, introducing a single insertion of length $k$ typically arises from one mutational event (e.g. replication slippage), rather than $k$ independent 1-bp insertions.
Therefore, biological alignments use **affine gap penalties**:
$$\text{Penalty}(k) = \text{open\_gap\_score} + (k - 1) \times \text{extend\_gap\_score}$$
where opening a gap is penalized heavily (e.g. $-10$), but extending an existing gap is penalized lightly (e.g. $-1$).


In [ ]:
# Worked Example: Comparing Linear vs Affine Gaps on HBB vs Myoglobin
from Bio import SeqIO
from Bio.Align import PairwiseAligner

hbb_prot = SeqIO.read(DATA_DIR / "sequences" / "hbb_human_protein.fasta", "fasta").seq
globin_seqs = {rec.id: rec.seq for rec in SeqIO.parse(DATA_DIR / "sequences" / "globin_family_proteins.fasta", "fasta")}
mb_prot = globin_seqs.get("MB_HUMAN", hbb_prot)

aligner = PairwiseAligner()
aligner.substitution_matrix = blosum62

# Linear gap penalty: open = -10, extend = -10
aligner.open_gap_score = -10
aligner.extend_gap_score = -10
linear_score = aligner.score(hbb_prot, mb_prot)

# Affine gap penalty: open = -10, extend = -1
aligner.open_gap_score = -10
aligner.extend_gap_score = -1
affine_score = aligner.score(hbb_prot, mb_prot)

print(f"HBB vs Myoglobin (BLOSUM62):")
print(f"  Score with Linear Gap (-10 / -10): {linear_score:.1f}")
print(f"  Score with Affine Gap (-10 / -1):   {affine_score:.1f}")


## 6. Compare with Web-Tool Practical
In Practical S6 (NCBI BLAST), the default scoring parameters are:
- Matrix: `BLOSUM62`
- Gap Costs: `Existence: 11, Extension: 1` (affine)
Notice that Biopython's `PairwiseAligner` uses negative numbers for penalties (`open_gap_score = -11`, `extend_gap_score = -1`), which matches NCBI's underlying score system.


## 7. Summary
- Smith–Waterman sets negative scores to zero, identifying localized regions of high similarity.
- BLOSUM62 is the general default; higher BLOSUM numbers correspond to closer sequences, whereas higher PAM numbers correspond to more distant sequences.
- Affine gap penalties penalize gap opening more than gap extension, reflecting evolutionary mechanisms.


## 8. Next Steps
- Read about database searching heuristics and E-values in `U05_C3_Reading_BLAST_Results.ipynb`.
- Practice aligning protein pairs with different scoring parameters in `U05_P_Sequence_Alignment.ipynb`.
